# 📊 Website Traffic Analysis — Alfido Tech Internship
**Dataset:** [Kaggle – bhanupratapbiswas/website-traffic-analysis](https://www.kaggle.com/datasets/bhanupratapbiswas/website-traffic-analysis)  
**Period:** August 19–25, 2021 &nbsp;|&nbsp; **Records:** 226,278  
---

## Step 1 — Install & Import Libraries

In [ ]:
# Run this cell first to install any missing libraries
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "pandas", "matplotlib", "seaborn", "openpyxl", "--quiet"])

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 20)
%matplotlib inline

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("✅ All libraries ready!")

## Step 2 — Load Dataset
> ⚠️ Make sure `traffic.xlsx` is in the **same folder** as this notebook.

In [ ]:
df = pd.read_excel("traffic.xlsx", parse_dates=["date"])

print(f"Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Dates  : {df['date'].min().date()}  →  {df['date'].max().date()}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

## Step 3 — Data Types

In [ ]:
df.info()

## Step 4 — Summary Statistics

In [ ]:
df.describe(include="all")

## Step 5 — Missing Values

In [ ]:
missing = pd.DataFrame({
    "dtype":     df.dtypes,
    "non_null":  df.count(),
    "missing":   df.isnull().sum(),
    "missing_%": (df.isnull().sum() / len(df) * 100).round(3),
    "unique":    df.nunique(),
    "mode":      [df[c].mode()[0] if df[c].notna().any() else "N/A" for c in df.columns],
})
missing

## Step 6 — Clean Data

In [ ]:
df["country"]  = df["country"].fillna("Unknown")
df["city"]     = df["city"].fillna("Unknown")
df["artist"]   = df["artist"].fillna("Unknown")
df["track"]    = df["track"].fillna("Unknown")
df["day"]      = df["date"].dt.date
df["weekday"]  = df["date"].dt.day_name()

print("Missing values after cleaning:")
print(df.isnull().sum())
print(f"\n✅ {len(df):,} rows ready for analysis")

## Step 7 — Key Metrics

In [ ]:
total     = len(df)
pageviews = (df["event"] == "pageview").sum()
clicks    = (df["event"] == "click").sum()
previews  = (df["event"] == "preview").sum()
ctr       = clicks / pageviews * 100

print("=" * 42)
print(f"  Total Events       : {total:>10,}")
print(f"  Pageviews          : {pageviews:>10,}  ({pageviews/total*100:.1f}%)")
print(f"  Clicks             : {clicks:>10,}  ({clicks/total*100:.1f}%)")
print(f"  Previews           : {previews:>10,}  ({previews/total*100:.1f}%)")
print(f"  Click-Through Rate : {ctr:>9.1f}%")
print(f"  Countries          : {df['country'].nunique():>10,}")
print(f"  Unique Artists     : {df['artist'].nunique():>10,}")
print(f"  Unique Tracks      : {df['track'].nunique():>10,}")
print("=" * 42)

## Step 8 — Chart 1: Event Type Distribution (Bar Chart)

In [ ]:
event_counts = df["event"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = ["#2563EB", "#059669", "#D97706"]
bars = ax.bar(event_counts.index, event_counts.values,
              color=bar_colors, width=0.5, zorder=3)

for bar, val in zip(bars, event_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1500,
            f"{val:,}", ha="center", va="bottom",
            fontsize=10, fontweight="bold", color="#1e293b")

ax.set_title("Event Type Distribution", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Number of Events", fontsize=10)
ax.set_xlabel("Event Type", fontsize=10)
ax.set_ylim(0, event_counts.max() * 1.18)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## Step 9 — Chart 2: Daily Traffic Trend (Line Chart)

In [ ]:
daily = df.groupby("day").size().reset_index(name="events")
daily["label"] = pd.to_datetime(daily["day"]).dt.strftime("%b %d")

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(daily["label"], daily["events"], alpha=0.12, color="#2563EB")
ax.plot(daily["label"], daily["events"],
        color="#2563EB", linewidth=2.5,
        marker="o", markersize=8,
        markerfacecolor="white", markeredgewidth=2.5,
        markeredgecolor="#2563EB", zorder=5)

for _, row in daily.iterrows():
    ax.annotate(f"{int(row['events']):,}",
                xy=(row["label"], row["events"]),
                xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=9,
                color="#334155", fontweight="bold")

ax.set_title("Daily Traffic Trend — Aug 19–25, 2021",
             fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Total Events", fontsize=10)
ax.set_xlabel("Date", fontsize=10)
ax.set_ylim(daily["events"].min() * 0.85, daily["events"].max() * 1.13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

peak = daily.loc[daily["events"].idxmax()]
low  = daily.loc[daily["events"].idxmin()]
print(f"Peak    : {peak['label']}  →  {int(peak['events']):,} events")
print(f"Lowest  : {low['label']}  →  {int(low['events']):,} events")
print(f"Mid-week drop: {(peak['events'] - low['events']) / peak['events'] * 100:.1f}%")

## Step 10 — Chart 3: Top 10 Countries (Horizontal Bar Chart)

In [ ]:
top_countries = df["country"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 5))
colors_c = ["#1E40AF"] + ["#93C5FD"] * 9
bars = ax.barh(top_countries.index[::-1],
               top_countries.values[::-1],
               color=colors_c[::-1], height=0.6, zorder=3)

for bar, val in zip(bars, top_countries.values[::-1]):
    ax.text(val + 300,
            bar.get_y() + bar.get_height() / 2,
            f"{val:,}  ({val/total*100:.1f}%)",
            va="center", fontsize=8.5,
            color="#334155", fontweight="bold")

ax.set_title("Top 10 Countries by Traffic", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Number of Events", fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xlim(0, top_countries.max() * 1.25)
plt.tight_layout()
plt.show()

## Step 11 — Chart 4: Event Share (Donut Pie Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sizes  = [pageviews, clicks, previews]
labels = ["Pageview", "Click", "Preview"]
colors_p = ["#2563EB", "#059669", "#D97706"]

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors_p,
    autopct="%1.1f%%", startangle=90,
    wedgeprops=dict(width=0.6, edgecolor="white", linewidth=2),
    pctdistance=0.75, textprops={"fontsize": 11}
)
for at in autotexts:
    at.set_color("white")
    at.set_fontweight("bold")

ax.set_title("Event Type Share", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

## Step 12 — Chart 5: Top 10 Tracks

In [ ]:
top_tracks = df["track"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 4.5))
colors_t = ["#1E40AF"] + ["#60A5FA"] * 9
bars = ax.bar(range(len(top_tracks)), top_tracks.values,
              color=colors_t, width=0.6, zorder=3)

for bar, val in zip(bars, top_tracks.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 300,
            f"{val:,}", ha="center", va="bottom",
            fontsize=8.5, fontweight="bold", color="#334155")

ax.set_xticks(range(len(top_tracks)))
ax.set_xticklabels(
    [t[:22] + "…" if len(t) > 22 else t for t in top_tracks.index],
    rotation=30, ha="right", fontsize=9
)
ax.set_title("Top 10 Tracks by Total Events", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Events", fontsize=10)
ax.set_ylim(0, top_tracks.max() * 1.15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## Step 13 — Chart 6: Event Types by Top 5 Countries (Stacked Bar)

In [ ]:
top5 = df["country"].value_counts().head(5).index.tolist()
grp  = (df[df["country"].isin(top5)]
        .groupby(["country", "event"])
        .size()
        .unstack(fill_value=0)
        .loc[top5])

fig, ax = plt.subplots(figsize=(9, 4.5))
grp.plot(kind="bar", ax=ax,
         color=["#2563EB", "#059669", "#D97706"],
         width=0.6, zorder=3,
         edgecolor="white", linewidth=0.5)

ax.set_title("Event Types — Top 5 Countries", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Country", fontsize=10)
ax.set_ylabel("Events", fontsize=10)
ax.tick_params(axis="x", rotation=15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(title="Event Type", fontsize=9)
plt.tight_layout()
plt.show()

## Step 14 — Missing Values Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.5))
sns.heatmap(df.isnull().T,
            cmap=["#F0FDF4", "#EF4444"],
            cbar=False, ax=ax,
            linewidths=0, xticklabels=False)
ax.set_title("Missing Values Map  (red = missing)",
             fontsize=13, fontweight="bold", pad=10)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)
plt.tight_layout()
plt.show()

print("Missing counts:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## Step 15 — Bounce Rate Proxy

In [ ]:
sessions   = df.groupby(["linkid", "day"])["event"].agg(list).reset_index()
total_s    = len(sessions)
converted  = sessions["event"].apply(lambda x: "click" in x).sum()
bounced    = sessions["event"].apply(lambda x: x == ["pageview"]).sum()

print("=" * 42)
print(f"  Approx. Sessions   : {total_s:>10,}")
print(f"  Sessions w/ Click  : {converted:>10,}  ({converted/total_s*100:.1f}%)")
print(f"  Single-PV Bounced  : {bounced:>10,}  ({bounced/total_s*100:.1f}%)")
print("=" * 42)

## Step 16 — Key Insights & Recommendations

### 📌 Insight 1 — MENA Region Dominates (20.9% from Saudi Arabia alone)
- **Action:** Build Arabic RTL landing pages. Target MENA with geo-specific campaigns.  
- **Expected impact:** 10–15% conversion lift in the largest traffic segment.

### 📌 Insight 2 — Preview → Click Drop-Off (28,531 unconverted previews)
- **Action:** A/B test richer preview cards — 30-second audio clips, artist bios, social proof badges.  
- **Expected impact:** +5% preview-to-click rate = ~1,400 extra weekly clicks.

### 📌 Insight 3 — Mid-Week Traffic Dip (~16% drop by Wednesday)
- **Action:** Schedule content drops and push notifications on Wednesday–Thursday.  
- **Expected impact:** Smoother weekly revenue curve.

### 📌 Insight 4 — Single-Track Risk ("Jalebi Baby" = 18.1% of all events)
- **Action:** Diversify promoted playlists. Surface Anne-Marie, Tundra Beats, Olivia Rodrigo.

### 📌 Insight 5 — ISRC Data Quality (3.15% missing)
- **Action:** Add ingestion-layer validation to require ISRC on every new content record.


## Step 17 — Export Summary CSVs

In [ ]:
df["event"].value_counts().to_csv("event_summary.csv", header=["count"])
df["country"].value_counts().head(20).to_csv("country_summary.csv", header=["count"])
df["track"].value_counts().head(20).to_csv("top_tracks.csv", header=["count"])

daily_out = df.groupby("day").size().reset_index(name="events")
daily_out.to_csv("daily_traffic.csv", index=False)

print("✅ Exported:")
print("   event_summary.csv")
print("   country_summary.csv")
print("   top_tracks.csv")
print("   daily_traffic.csv")